In [1]:
# =====================================================================
# Task-Level Features Show No Detectable Risk Signal in a Public
# Construction Dataset: A Diagnostic Audit
#
# Full reproducible analysis. Regenerates every number in the paper.
# Python 3.12 | scikit-learn | XGBoost | statsmodels | seed = 42
# Dataset: Kaggle "Construction Project Management Dataset" (CC0)
# =====================================================================

from google.colab import files
print("Upload construction_dataset.csv")
uploaded = files.upload()

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.datasets import make_classification
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ------------------------------------------------------------ 1. DATA
df = pd.read_csv('construction_dataset.csv')
X  = df.drop(columns=['Task_ID', 'Risk_Level'])
le = LabelEncoder(); y = le.fit_transform(df['Risk_Level'])
CLASSES, FEATURES = list(le.classes_), list(X.columns)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                      random_state=SEED, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
N = len(yte)

print("\n" + "="*72); print("1. DATASET"); print("="*72)
print(f"Shape {df.shape} | Classes {CLASSES} | Train/Test {len(ytr)}/{N}")
vc = df['Risk_Level'].value_counts()
for c in ['Low','Medium','High']:
    print(f"  {c:7s} {vc[c]:4d} ({vc[c]/len(df)*100:.1f}%)")
print(f"Missing {df.isnull().sum().sum()} | Duplicates {df.duplicated().sum()}")
print(f"Task_Duration_Days {df.Task_Duration_Days.min()}-{df.Task_Duration_Days.max()}")
print(f"Material_Cost_USD ${df.Material_Cost_USD.min():,.0f}-${df.Material_Cost_USD.max():,.0f}")

# ---------------------------------------------------- 2. CORRELATIONS
print("\n" + "="*72); print("2. FEATURE-LABEL CORRELATION"); print("="*72)
corr = {f: np.corrcoef(X[f], y)[0,1] for f in FEATURES}
for f, v in sorted(corr.items(), key=lambda kv: -abs(kv[1])):
    print(f"  {f:26s} r = {v:+.4f}")
print(f"  Max |r| = {max(abs(v) for v in corr.values()):.4f}")

# --------------------------------------------------------- 3. HELPERS
def ev(name, model, weighted=False, quiet=False):
    if weighted:
        model.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', y=ytr))
    else:
        model.fit(Xtr, ytr)
    p = model.predict(Xte)
    r = dict(pred=p,
             acc=accuracy_score(yte, p),
             mf1=f1_score(yte, p, average='macro', zero_division=0),
             mp=precision_score(yte, p, average='macro', zero_division=0),
             mr=recall_score(yte, p, average='macro', zero_division=0),
             hr=recall_score(yte, p, average=None, zero_division=0)[CLASSES.index('High')])
    if not quiet:
        print(f"\n--- {name} ---")
        print(f"  Accuracy {r['acc']*100:.2f}% ({(p==yte).sum()}/{N}) | "
              f"macro P/R/F1 {r['mp']:.3f}/{r['mr']:.3f}/{r['mf1']:.3f} | "
              f"High recall {r['hr']:.3f}")
        print("  Confusion (rows=true " + str(CLASSES) + "):")
        print("  " + str(confusion_matrix(yte, p)).replace("\n", "\n  "))
    return r

R = {}

# ------------------------------------------------- 4. INITIAL MODELS
print("\n" + "="*72); print("4. INITIAL MODELS (untuned, unweighted)"); print("="*72)
R['lr0']  = ev("Logistic Regression (default)",
               make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)))
R['rf0']  = ev("Random Forest (100 trees, depth 10)",
               RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1))
R['xgb0'] = ev("XGBoost (200 est, depth 5, lr 0.3)",
               XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.3,
                             random_state=SEED, verbosity=0))

# ------------------------------------------------- 5. CLASS-WEIGHTED
print("\n" + "="*72); print("5. CLASS-WEIGHTED MODELS (untuned)"); print("="*72)
R['rfw']  = ev("Random Forest (class-weighted)",
               RandomForestClassifier(n_estimators=100, max_depth=10,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1))
R['xgbw'] = ev("XGBoost (class-weighted)",
               XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.3,
                             random_state=SEED, verbosity=0), weighted=True)

# ------------------------------------------------------------- 6. CV
print("\n" + "="*72); print("6. 5-FOLD CROSS-VALIDATION"); print("="*72)
for nm, m in [
    ("LR (default)",           make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))),
    ("RF unweighted (100/10)", RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1)),
    ("RF weighted (100/10)",   RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=SEED, n_jobs=-1)),
    ("XGB unweighted",         XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.3, random_state=SEED, verbosity=0)),
]:
    s = cross_val_score(m, X, y, cv=cv, scoring='accuracy')
    print(f"  {nm:26s} {s.mean()*100:.2f}% (± {s.std()*100:.2f}%)")

# ------------------------------------------ 7. GRIDSEARCH (accuracy)
print("\n" + "="*72); print("7. GRIDSEARCH (accuracy scoring)"); print("="*72)
GA = {}
GA['LR'] = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
                        {'logisticregression__C':[0.001,0.01,0.1,1,10,100]},
                        cv=cv, scoring='accuracy', n_jobs=-1).fit(Xtr, ytr)
GA['RF'] = GridSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=-1),
                        {'n_estimators':[50,100,200],'max_depth':[5,10,15],'min_samples_split':[2,5,10]},
                        cv=cv, scoring='accuracy', n_jobs=-1).fit(Xtr, ytr)
GA['XGB']= GridSearchCV(XGBClassifier(random_state=SEED, verbosity=0),
                        {'n_estimators':[100,200,300],'max_depth':[3,5,7],'learning_rate':[0.01,0.1,0.3]},
                        cv=cv, scoring='accuracy', n_jobs=-1).fit(Xtr, ytr)
for k in ['LR','RF','XGB']:
    print(f"  {k:4s} best={GA[k].best_params_} CV={GA[k].best_score_*100:.2f}%")
for k in ['LR','RF','XGB']:
    R[f'{k}_t'] = ev(f"{k} tuned", GA[k].best_estimator_)

# ------------------------------------------ 8. GRIDSEARCH (macro-F1)
print("\n" + "="*72); print("8. GRIDSEARCH (macro-F1 scoring)"); print("="*72)
GF = {}
GF['RF'] = GridSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=-1),
                        {'n_estimators':[50,100,200],'max_depth':[5,10,15],'min_samples_split':[2,5,10]},
                        cv=cv, scoring='f1_macro', n_jobs=-1).fit(Xtr, ytr)
GF['XGB']= GridSearchCV(XGBClassifier(random_state=SEED, verbosity=0),
                        {'n_estimators':[100,200,300],'max_depth':[3,5,7],'learning_rate':[0.01,0.1,0.3]},
                        cv=cv, scoring='f1_macro', n_jobs=-1).fit(Xtr, ytr)
for k in ['RF','XGB']:
    print(f"  {k:4s} best={GF[k].best_params_} CV macro-F1={GF[k].best_score_:.3f}")
    R[f'{k}_f1'] = ev(f"{k} F1-optimized", GF[k].best_estimator_)

# --------------------------------------------- 9. FEATURE IMPORTANCE
print("\n" + "="*72); print("9. FEATURE IMPORTANCE"); print("="*72)
gain = dict(zip(FEATURES, GA['XGB'].best_estimator_.feature_importances_))
print("XGBoost gain-based:")
for f, v in sorted(gain.items(), key=lambda kv: -kv[1]):
    print(f"  {f:26s} {v:.4f}")
print(f"  Range {max(gain.values()):.4f} to {min(gain.values()):.4f}")

pi = permutation_importance(GA['RF'].best_estimator_, Xte, yte,
                            n_repeats=30, random_state=SEED, n_jobs=-1)
print("\nPermutation importance (tuned RF, 30 repeats):")
for i in np.argsort(-pi.importances_mean):
    print(f"  {FEATURES[i]:26s} {pi.importances_mean[i]:+.4f} ± {pi.importances_std[i]:.4f}")

# -------------------------------------------- 10. FEATURE ENGINEERING
print("\n" + "="*72); print("10. FEATURE ENGINEERING"); print("="*72)
Xe = X.copy()
Xe['Duration_x_Labor']   = X.Task_Duration_Days * X.Labor_Required
Xe['Cost_x_Resource']    = X.Material_Cost_USD  * X.Resource_Constraint_Score
Xe['Cost_x_Site']        = X.Material_Cost_USD  * X.Site_Constraint_Score
Xe['Duration_sq']        = X.Task_Duration_Days ** 2
Xe['Constraint_Sum']     = X.Resource_Constraint_Score + X.Site_Constraint_Score
Xe['Constraint_Product'] = X.Resource_Constraint_Score * X.Site_Constraint_Score
for k in ['RF','XGB']:
    b = cross_val_score(GA[k].best_estimator_, X,  y, cv=cv, scoring='accuracy').mean()
    e = cross_val_score(GA[k].best_estimator_, Xe, y, cv=cv, scoring='accuracy').mean()
    print(f"  {k:4s} baseline CV {b*100:.2f}% -> engineered CV {e*100:.2f}% ({(e-b)*100:+.2f} pts)")

# ------------------------------------------------- 11. ERROR ANALYSIS
print("\n" + "="*72); print("11. ERROR ANALYSIS (class-weighted RF)"); print("="*72)
ok = (R['rfw']['pred'] == yte); Xr = Xte.reset_index(drop=True)
print(f"  Correct {ok.sum()} | Incorrect {(~ok).sum()}\n")
print(f"  {'Feature':26s} {'Correct':>11s} {'Incorrect':>11s} {'Diff':>10s}")
for f in FEATURES:
    a, b = Xr[f][ok].mean(), Xr[f][~ok].mean()
    print(f"  {f:26s} {a:11.2f} {b:11.2f} {b-a:+10.2f}")

# ------------------------------------- 12. BASELINES & SIGNIFICANCE
print("\n" + "="*72); print("12. BASELINES & SIGNIFICANCE"); print("="*72)
print("Reference levels:")
for s, lab in [('most_frequent','Majority-class baseline'),
               ('stratified','Stratified random guesser'),
               ('uniform','Uniform random guesser')]:
    d = DummyClassifier(strategy=s, random_state=SEED).fit(Xtr, ytr).predict(Xte)
    print(f"  {lab:26s} acc {accuracy_score(yte,d)*100:5.2f}%  "
          f"macro-F1 {f1_score(yte,d,average='macro',zero_division=0):.3f}")

maj = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr).predict(Xte)
mj  = (maj == yte)
print(f"\n{'Model':14s} {'Acc':>8s} {'95% Wilson CI':>20s} {'McNemar p':>11s}")
for k, lab in [('LR_t','LR tuned'), ('RF_t','RF tuned'), ('XGB_t','XGB tuned')]:
    mc = (R[k]['pred'] == yte); c = int(mc.sum())
    lo, hi = proportion_confint(c, N, alpha=0.05, method='wilson')
    tbl = [[int((mc&mj).sum()), int((mc&~mj).sum())],
           [int((~mc&mj).sum()), int((~mc&~mj).sum())]]
    print(f"{lab:14s} {c/N*100:7.2f}%  [{lo*100:5.1f}%, {hi*100:5.1f}%] "
          f"{mcnemar(tbl, exact=True).pvalue:11.4f}")
c = int(mj.sum()); lo, hi = proportion_confint(c, N, alpha=0.05, method='wilson')
print(f"{'Majority':14s} {c/N*100:7.2f}%  [{lo*100:5.1f}%, {hi*100:5.1f}%] {'—':>11s}")

# ---------------------------------------------- 13. PERMUTATION TEST
print("\n" + "="*72); print("13. LABEL PERMUTATION TEST (1000 shuffles)"); print("="*72)
bp  = GA['RF'].best_params_
obs = R['RF_t']['acc']
rng, null = np.random.default_rng(SEED), []
for i in range(1000):
    m = RandomForestClassifier(**bp, random_state=SEED, n_jobs=-1)
    m.fit(Xtr, rng.permutation(ytr))
    null.append(accuracy_score(yte, m.predict(Xte)))
    if (i+1) % 250 == 0: print(f"  ...{i+1}/1000")
null = np.array(null)
p_perm = (np.sum(null >= obs) + 1) / (len(null) + 1)
print(f"\n  Observed {obs*100:.2f}% | Null mean {null.mean()*100:.2f}% "
      f"(SD {null.std()*100:.2f}%) | 95th pct {np.percentile(null,95)*100:.2f}%")
print(f"  Permutation p = {p_perm:.4f}")

# -------------------------- 14. POSITIVE CONTROL & SENSITIVITY SWEEP
print("\n" + "="*72); print("14. POSITIVE CONTROL — SENSITIVITY SWEEP"); print("="*72)
print("Does the diagnostic sequence detect signal where signal exists?\n")
print(f"{'class_sep':>10} {'max|r|':>8} {'RF acc':>9} {'margin':>9} "
      f"{'macro-F1':>9} {'High rec':>9} {'perm p':>8}")
print("-"*70)
for sep in [0.02,0.04,0.06,0.08,0.10,0.20,0.30,0.50,0.75,1.00]:
    Xs, ys = make_classification(n_samples=1300, n_features=8, n_informative=5,
                                 n_redundant=0, n_classes=3,
                                 weights=[0.508,0.290,0.202],
                                 class_sep=sep, random_state=SEED)
    Xs = pd.DataFrame(Xs, columns=[f'f{i}' for i in range(8)])
    a, b_, c_, d_ = train_test_split(Xs, ys, test_size=0.2, random_state=SEED, stratify=ys)
    mr_ = max(abs(np.corrcoef(Xs[c][:], ys)[0,1]) for c in Xs.columns)
    m = RandomForestClassifier(**bp, random_state=SEED, n_jobs=-1).fit(a, c_)
    pr = m.predict(b_)
    acc_ = accuracy_score(d_, pr)
    base_= accuracy_score(d_, DummyClassifier(strategy='most_frequent').fit(a,c_).predict(b_))
    rg, nl = np.random.default_rng(SEED), []
    for _ in range(200):
        mm = RandomForestClassifier(**bp, random_state=SEED, n_jobs=-1)
        mm.fit(a, rg.permutation(c_))
        nl.append(accuracy_score(d_, mm.predict(b_)))
    nl = np.array(nl); pv = (np.sum(nl >= acc_)+1)/(len(nl)+1)
    print(f"{sep:>10.2f} {mr_:>8.4f} {acc_*100:>8.2f}% {(acc_-base_)*100:>+8.2f} "
          f"{f1_score(d_,pr,average='macro'):>9.3f} "
          f"{recall_score(d_,pr,average=None,zero_division=0)[2]:>9.3f} {pv:>8.4f}")
print("-"*70)
print(f"{'REAL DATA':>10} {0.0596:>8.4f} {obs*100:>8.2f}% {+0.77:>+8.2f} "
      f"{R['RF_t']['mf1']:>9.3f} {R['RF_t']['hr']:>9.3f} {p_perm:>8.4f}")

print("\n" + "="*72); print("ANALYSIS COMPLETE"); print("="*72)

Upload construction_dataset.csv


Saving construction_dataset.csv to construction_dataset.csv

1. DATASET
Shape (1300, 10) | Classes ['High', 'Low', 'Medium'] | Train/Test 1040/260
  Low      660 (50.8%)
  Medium   377 (29.0%)
  High     263 (20.2%)
Missing 0 | Duplicates 0
Task_Duration_Days 1-89
Material_Cost_USD $1,003-$99,956

2. FEATURE-LABEL CORRELATION
  Start_Constraint           r = -0.0596
  Resource_Constraint_Score  r = +0.0372
  Dependency_Count           r = +0.0336
  Site_Constraint_Score      r = +0.0243
  Labor_Required             r = -0.0174
  Material_Cost_USD          r = +0.0170
  Task_Duration_Days         r = -0.0101
  Equipment_Units            r = -0.0054
  Max |r| = 0.0596

4. INITIAL MODELS (untuned, unweighted)

--- Logistic Regression (default) ---
  Accuracy 50.77% (132/260) | macro P/R/F1 0.169/0.333/0.224 | High recall 0.000
  Confusion (rows=true ['High', 'Low', 'Medium']):
  [[  0  53   0]
   [  0 132   0]
   [  0  75   0]]

--- Random Forest (100 trees, depth 10) ---
  Accuracy 50.77